In [11]:
import numpy as np
import pandas as pd
import folium
from branca.colormap import linear
from tqdm import tqdm

In [12]:
clients = pd.read_excel('../../data/data.xlsx', sheet_name='zips')

In [13]:
df_clients = clients[['Order Postal Code', 'Latitude', 'Longitude']]

In [14]:
# Bounding box of São Paulo
latitude_min = clients['Latitude'].min()
latitude_max = clients['Latitude'].max()
longitude_min = clients['Longitude'].min()
longitude_max = clients['Longitude'].max()

# Grid cell size in degrees (approximately 1 km)
cell_size = 0.009

# Number of grid cells in latitude and longitude
n_cells_lat = int(np.ceil((latitude_max - latitude_min) / cell_size))
n_cells_lon = int(np.ceil((longitude_max - longitude_min) / cell_size))

# Create map centered on São Paulo
map_ = folium.Map(location=[(latitude_min + latitude_max) / 2, (longitude_min + longitude_max) / 2], zoom_start=12)

df_grids = pd.DataFrame(columns=['id', 'maxLat', 'minLat', 'maxLong', 'minLong'])

# Iterate over grid cells in latitude and longitude
i = 0
for lat in np.arange(latitude_min, latitude_min + n_cells_lat * cell_size, cell_size):
    for lon in np.arange(longitude_min, longitude_min + n_cells_lon * cell_size, cell_size):
            sw = [lat, lon]
            ne = [lat + cell_size, lon + cell_size]
            bounds = [sw, ne]
            folium.Rectangle(bounds, color='black', fill=False).add_to(map_)
            df_grids.at[i, 'minLat'] = lat
            df_grids.at[i, 'minLong'] = lon
            df_grids.at[i, 'maxLat'] = lat + cell_size
            df_grids.at[i, 'maxLong'] = lon + cell_size
            df_grids.at[i, 'id'] = i
            i = i+1

# Display map with grid cells
map_


In [15]:
# Optimized function to assign a client to a grid cell
def assign_client(client, cells_dict):
    global pbar
    pbar.update(1)
    for _, cell in cells_dict.items():
        if (
            cell['minLat'] <= client['Latitude'] <= cell['maxLat'] and
            cell['minLong'] <= client['Longitude'] <= cell['maxLong']
        ):
            return cell['id']
    return None

# Convert 'grids' DataFrame to a dictionary
cells_dict = {row['id']: row for _, row in df_grids.iterrows()}

# Add column to assign clients to grid cells
with tqdm(total=len(df_clients)) as pbar:
    df_clients['Grid'] = df_clients.apply(lambda row: assign_client(row, cells_dict), axis=1)
    
df_result = df_clients.merge(df_grids, left_on='Grid', right_on='id').drop('id', axis=1)

100%|██████████| 35473/35473 [02:52<00:00, 205.43it/s]


In [16]:
df_result.to_csv('../../results/utils/df_clients_square.csv')

In [17]:
df_grids.to_csv('../../results/utils/df_clients_square.csv')